In [1]:
from pathlib import Path
import pandas as pd

current_folder = Path.cwd()
project_root = (
    current_folder.parent
    if current_folder.name == "notebooks"
    else current_folder
)

planning_households = pd.read_parquet(
    project_root / "data" / "processed"
    / "campaign_18_household_analysis.parquet",
    columns=["household_key", "sales_before"]
)

print(f"Households available: {len(planning_households):,}")
print(
    f"Mean 56-day retailer sales: "
    f"{planning_households['sales_before'].mean():.2f}"
)
print(
    f"Standard deviation of 56-day retailer sales: "
    f"{planning_households['sales_before'].std(ddof=1):.2f}"
)

Households available: 2,498
Mean 56-day retailer sales: 274.30
Standard deviation of 56-day retailer sales: 328.46


In [2]:
baseline_mean = planning_households["sales_before"].mean()
baseline_sd = planning_households["sales_before"].std(ddof=1)

relative_mde = 0.10
absolute_mde = baseline_mean * relative_mde

alpha = 0.05
desired_power = 0.80

print(f"Baseline mean sales: {baseline_mean:.2f}")
print(f"Detectable relative increase: {relative_mde:.0%}")
print(f"Detectable absolute difference: {absolute_mde:.2f}")

Baseline mean sales: 274.30
Detectable relative increase: 10%
Detectable absolute difference: 27.43


In [3]:
import math
from statsmodels.stats.power import TTestIndPower

standardized_effect = absolute_mde / baseline_sd

power_analysis = TTestIndPower()

required_per_group = math.ceil(
    power_analysis.solve_power(
        effect_size=standardized_effect,
        alpha=alpha,
        power=desired_power,
        ratio=1.0,
        alternative="two-sided"
    )
)

required_total = required_per_group * 2

print(f"Required households per group: {required_per_group:,}")
print(f"Required households in total: {required_total:,}")
print(f"Households available in our planning cohort: {len(planning_households):,}")

Required households per group: 2,252
Required households in total: 4,504
Households available in our planning cohort: 2,498


In [4]:
available_per_group = len(planning_households) // 2

achievable_power = power_analysis.power(
    effect_size=standardized_effect,
    nobs1=available_per_group,
    alpha=alpha,
    ratio=1.0,
    alternative="two-sided"
)

print(f"Available households per group: {available_per_group:,}")
print(f"Power to detect the planned 10% difference: {achievable_power:.2%}")

Available households per group: 1,249
Power to detect the planned 10% difference: 55.02%


In [5]:
detectable_standardized_effect = power_analysis.solve_power(
    effect_size=None,
    nobs1=available_per_group,
    alpha=alpha,
    power=desired_power,
    ratio=1.0,
    alternative="two-sided"
)

detectable_absolute_difference = (
    detectable_standardized_effect * baseline_sd
)

detectable_relative_difference = (
    detectable_absolute_difference / baseline_mean
)

print(f"Detectable absolute difference: {detectable_absolute_difference:.2f}")
print(f"Detectable relative difference: {detectable_relative_difference:.2%}")

Detectable absolute difference: 36.84
Detectable relative difference: 13.43%


In [6]:
ab_report_folder = project_root / "reports" / "ab_test_design"
ab_report_folder.mkdir(parents=True, exist_ok=True)

power_summary = pd.DataFrame([{
    "outcome_window_days": 56,
    "baseline_mean": baseline_mean,
    "assumed_standard_deviation": baseline_sd,
    "alpha": alpha,
    "target_power": desired_power,
    "planned_relative_difference": relative_mde,
    "planned_absolute_difference": absolute_mde,
    "required_per_group": required_per_group,
    "required_total": required_total,
    "available_per_group": available_per_group,
    "power_with_available_sample": achievable_power,
    "detectable_absolute_difference": detectable_absolute_difference,
    "detectable_relative_difference": detectable_relative_difference
}])

power_summary.to_csv(
    ab_report_folder / "power_analysis.csv",
    index=False
)

print("Saved power-analysis assumptions and results.")

Saved power-analysis assumptions and results.


In [7]:
import numpy as np

assert planning_households["household_key"].is_unique

simulated_assignment = (
    planning_households
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

simulated_assignment["assigned_group"] = np.where(
    simulated_assignment.index < available_per_group,
    "campaign",
    "holdout"
)

simulated_assignment["is_simulated_assignment"] = True

simulated_assignment["assigned_group"].value_counts()

assigned_group
campaign    1249
holdout     1249
Name: count, dtype: int64

In [8]:
assignment_balance = simulated_assignment.groupby("assigned_group").agg(
    households=("household_key", "size"),
    mean_baseline_sales=("sales_before", "mean"),
    median_baseline_sales=("sales_before", "median"),
    baseline_sales_sd=("sales_before", "std")
)

assignment_balance.round(2)

,households,mean_baseline_sales,median_baseline_sales,baseline_sales_sd
assigned_group,,,,
campaign,1249,274.63,168.04,318.14
holdout,1249,273.97,153.42,338.59


In [9]:
simulation_folder = project_root / "data" / "simulated"
simulation_folder.mkdir(parents=True, exist_ok=True)

assignment_path = (
    simulation_folder / "simulated_campaign_assignment.parquet"
)

simulated_assignment.to_parquet(assignment_path, index=False)

assignment_balance.to_csv(
    ab_report_folder / "simulated_assignment_balance.csv",
    index=True
)

print(f"Saved simulated assignment: {assignment_path}")
print("Saved baseline balance summary.")

Saved simulated assignment: /Users/faridoonfarahi/Documents/Personal_Projects/customer_loyalty_platform/data/simulated/simulated_campaign_assignment.parquet
Saved baseline balance summary.


In [10]:
outcome_rng = np.random.default_rng(2026)

simulated_trial = simulated_assignment.copy()
true_simulated_effect = absolute_mde

simulated_no_campaign_sales = outcome_rng.choice(
    planning_households["sales_before"].to_numpy(),
    size=len(simulated_trial),
    replace=True
)

simulated_trial["simulated_sales_56d"] = (
    simulated_no_campaign_sales
    + np.where(
        simulated_trial["assigned_group"] == "campaign",
        true_simulated_effect,
        0.0
    )
)

simulated_trial["is_simulated_outcome"] = True

print(f"Imposed simulation effect: {true_simulated_effect:.2f}")

simulated_trial.groupby("assigned_group").agg(
    households=("household_key", "size"),
    mean_simulated_sales=("simulated_sales_56d", "mean"),
    simulated_sales_sd=("simulated_sales_56d", "std")
).round(2)

Imposed simulation effect: 27.43


,households,mean_simulated_sales,simulated_sales_sd
assigned_group,,,
campaign,1249,294.15,313.43
holdout,1249,287.26,346.28


In [11]:
from scipy.stats import ttest_ind

campaign_sales = simulated_trial.loc[
    simulated_trial["assigned_group"] == "campaign",
    "simulated_sales_56d"
]

holdout_sales = simulated_trial.loc[
    simulated_trial["assigned_group"] == "holdout",
    "simulated_sales_56d"
]

estimated_effect = campaign_sales.mean() - holdout_sales.mean()

welch_result = ttest_ind(
    campaign_sales,
    holdout_sales,
    equal_var=False
)

effect_interval = welch_result.confidence_interval(
    confidence_level=1 - alpha
)

print(f"Estimated simulated effect: {estimated_effect:.2f}")
print(
    f"95% confidence interval: "
    f"[{effect_interval.low:.2f}, {effect_interval.high:.2f}]"
)
print(f"Two-sided p-value: {welch_result.pvalue:.4f}")

Estimated simulated effect: 6.88
95% confidence interval: [-19.03, 32.80]
Two-sided p-value: 0.6025


In [12]:
simulation_results = pd.DataFrame([{
    "is_simulated": True,
    "imposed_effect": true_simulated_effect,
    "estimated_effect": estimated_effect,
    "ci_lower": effect_interval.low,
    "ci_upper": effect_interval.high,
    "p_value": welch_result.pvalue,
    "alpha": alpha,
    "reject_zero_effect": welch_result.pvalue < alpha,
    "campaign_households": len(campaign_sales),
    "holdout_households": len(holdout_sales),
    "assignment_seed": 42,
    "outcome_seed": 2026
}])

simulation_results.to_csv(
    ab_report_folder / "simulated_trial_results.csv",
    index=False
)

simulated_trial.to_parquet(
    simulation_folder / "simulated_campaign_trial.parquet",
    index=False
)

print("Saved simulated trial results and household outcomes.")

Saved simulated trial results and household outcomes.
